# PyTorch XLM-RoBERTa base

I will fine-tune `FacebookAI/xlm-roberta-base` with its matching tokenizer for three-way natural-language inference.

Reusable loading, splitting, batching, training, and evaluation code lives in `scripts/watson_model_helpers.py`.

The premise-grouped split evaluates generalization to unseen premises.

## Venv and requirements

- Using the existing local `kaggle_torch_py312` kernel.
- Transformers 4.44.2 and SentencePiece 0.2.2 are installed and verified with the existing PyTorch 2.2.2
- see requirements.txt for more info

## Model and tokenizer choice

- starting from general pretrained [XLM-RoBERTa base checkpoint](https://huggingface.co/FacebookAI/xlm-roberta-base)
- newly initialized three-label classification head. 
- The tokenizer receives premise first, hypothesis second, with dynamic batch padding and configurable truncation.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "watson_dataset_config.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from watson_dataset_helpers import load_datasets
from watson_model_helpers import (
    split_training_data,
    load_model_and_tokenizer,
    tokenize_data,
    make_loaders,
    fine_tune,
)

## Bring in the data

In [2]:
datasets = load_datasets()

training_data = datasets["train"].copy()
test_data = datasets["test"].copy()

Kaggle files already present; using: (project_root)/data/raw


## Training-data preparation and validation split

### Goals in Preparing Training/Validation split:
- No exact premise appears in both training and validation. All rows sharing a premise stay together.
- Have roughly the same label proportions in validation and the training side as a whole.

Note: Proportions are approximate because premise groups stay intact.

In [3]:
training, validation = split_training_data(training_data)

## Data Cleaning

Prior exploration in `01_dataset_exploration.ipynb` found

- no missing values
- no duplicate premise–hypothesis pairs

No further text cleaning is planned; the original premise and hypothesis text will be kept.

## Bring in Pre-Trained Model

### XLM-Roberta Base
- XLM-Roberta Base is chosen as the pre-trained model.
- XLM-Roberta seems like a natural first check.
- Base model chosen to baseline performance using, and acknowledging, local HW constraints
- Other models may be used in the future, either in this notebook or separate notebooks

In [4]:
model, tokenizer = load_model_and_tokenizer()

/Users/KurtMeehanPro/Documents/code/venv/kaggle_torch_py312/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Tokenization

In [5]:
training_encodings = tokenize_data(training, tokenizer)
validation_encodings = tokenize_data(validation, tokenizer)

## Batching, Padding, Training Settings

In [6]:
# Initial settings; not tuned.
batch_size = 1
learning_rate = 2e-5
epochs = 3

training_loader, validation_loader = make_loaders(
    training_encodings, validation_encodings, tokenizer, batch_size
)

## Fine-tuning

In [ ]:
fine_tune(model, training_loader, learning_rate, epochs)

## Evaluation

## Next observations
